# Fallbacks and Configurable Runnables

Production LLM apps face **transient provider errors**, **rate limits**, and **changing requirements** — swap models per tenant, tune temperature per route, or route to a cheaper backup when the primary fails. LangChain Runnables expose resilience via **`.with_fallbacks()`** and **`.with_retry()`**, and flexibility via **`ConfigurableField`** / **`configurable_alternatives`** passed through **`RunnableConfig`** (**02**, **15**).

This notebook covers fallback chains, retry policies, runtime model switching, configurable prompts and parsers, parser error fallbacks (**05**), agent **`ModelFallbackMiddleware`** (**13**), and production wiring patterns. The capstone resilience notebook before **17. Production Patterns for LangChain**.

Prerequisites: **02. Runnable Protocol and LCEL**, **05. Output Parsers and Structured Output**, **06. LCEL Composition Patterns**, **13. Agents with create_agent**, **15. Callbacks, Caching, and Observability**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json

import langchain_core

print("langchain-core:", langchain_core.__version__)

---

## 1. Two Production Concerns

| Concern | Question | LangChain tool |
|---------|----------|----------------|
| **Resilience** | What if this step fails? | `.with_retry()`, `.with_fallbacks()` |
| **Configurability** | Can I swap behavior per request? | `ConfigurableField`, `configurable_alternatives` |

```
Request + RunnableConfig
        │
        ├── configurable.llm  → pick model A vs B
        ├── configurable.temperature → tune per route
        │
        ▼
   Primary Runnable
        │ (on failure)
        ▼
   Fallback Runnable(s)
```

Both mechanisms use **`RunnableConfig`** — the same object that carries callbacks and tags (**15**).

---

## 2. Setup

Base chain components for resilience and configurability demos:

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer backend engineering questions in one or two sentences."),
    ("human", "{question}"),
])

primary_llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
backup_llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0.3)

base_chain = prompt | primary_llm | StrOutputParser()
print("base chain ready")

---

## 3. with_fallbacks — Backup Runnables

**`.with_fallbacks([r1, r2, ...])`** tries the primary Runnable; on exception, runs fallbacks **in order** until one succeeds:

In [ ]:
def flaky_upper(text: str) -> str:
    if "error" in text.lower():
        raise RuntimeError("simulated provider error")
    return text.upper()


robust_lambda = (
    RunnableLambda(flaky_upper)
    .with_fallbacks([RunnableLambda(lambda x: f"FALLBACK({x})")])
)

print(robust_lambda.invoke("ok"))
print(robust_lambda.invoke("trigger error"))

Same pattern applies to **LLM steps** — primary model fails → backup model handles the request.

In [ ]:
llm_with_fallback = primary_llm.with_fallbacks([backup_llm])
resilient_chain = prompt | llm_with_fallback | StrOutputParser()

print(resilient_chain.invoke({"question": "What is Alembic?"}))

### 3.1 Fallback Chain at the Runnable Level

Wrap the **entire chain** when any step might fail:

In [ ]:
simple_backup = RunnableLambda(
    lambda d: "Sorry — the assistant is temporarily unavailable. Please retry."
)

production_chain = base_chain.with_fallbacks([simple_backup])
print(production_chain.invoke({"question": "What is JWT?"}))

Use a **static string fallback** for user-facing outages; use a **backup model** when you still need a real answer.

---

## 4. with_retry — Transient Errors

**`.with_retry()`** re-invokes the **same** Runnable on failure — ideal for rate limits and network blips:

In [ ]:
attempt_counter = {"n": 0}


def flaky_once(text: str) -> str:
    attempt_counter["n"] += 1
    if attempt_counter["n"] == 1:
        raise ConnectionError("transient network blip")
    return f"success on attempt {attempt_counter['n']}: {text}"


retrying = RunnableLambda(flaky_once).with_retry(
    stop_after_attempt=3,
    wait_exponential_jitter=True,
)

attempt_counter["n"] = 0
print(retrying.invoke("hello"))

| Parameter | Meaning |
|-----------|--------|
| **`stop_after_attempt`** | Max tries (including first) |
| **`wait_exponential_jitter`** | Backoff between retries |
| **`retry_if_exception_type`** | Limit which exceptions retry |

**Retry then fallback** is a common stack: `.with_retry(...).with_fallbacks([backup])`.

In [ ]:
retry_then_fallback = (
    RunnableLambda(flaky_upper)
    .with_retry(stop_after_attempt=2)
    .with_fallbacks([RunnableLambda(lambda x: f"RECOVERY({x})")])
)

print(retry_then_fallback.invoke("trigger error"))

---

## 5. ConfigurableField — Runtime Parameters

Expose Runnable parameters that callers set via **`config["configurable"]`**:

In [ ]:
from langchain_core.runnables import ConfigurableField

configurable_llm = primary_llm.configurable_fields(
    temperature=ConfigurableField(
        id="temperature",
        name="LLM Temperature",
        description="Sampling temperature for this request",
    ),
    max_tokens=ConfigurableField(
        id="max_tokens",
        name="Max Tokens",
        description="Maximum tokens in the response",
    ),
)

configurable_chain = prompt | configurable_llm | StrOutputParser()

precise = configurable_chain.invoke(
    {"question": "Define idempotency briefly."},
    config={"configurable": {"temperature": 0, "max_tokens": 60}},
)
creative = configurable_chain.invoke(
    {"question": "Define idempotency briefly."},
    config={"configurable": {"temperature": 0.8, "max_tokens": 120}},
)

print("precise: ", precise)
print("creative:", creative)

Same chain, different **`configurable`** values per request — no code fork per route.

---

## 6. configurable_alternatives — Swap Entire Runnables

Switch between **whole implementations** at runtime — e.g. primary vs economy model:

In [ ]:
economy_llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
premium_llm = ChatOpenAI(model="gpt-4o", api_key=OPENAI_API_KEY, temperature=0)

routed_llm = primary_llm.configurable_alternatives(
    ConfigurableField(id="llm"),
    default_key="economy",
    alternatives={
        "economy": economy_llm,
        "premium": premium_llm,
    },
)

routed_chain = prompt | routed_llm | StrOutputParser()

economy_answer = routed_chain.invoke(
    {"question": "What is FastAPI?"},
    config={"configurable": {"llm": "economy"}},
)
premium_answer = routed_chain.invoke(
    {"question": "What is FastAPI?"},
    config={"configurable": {"llm": "premium"}},
)

print("economy:", economy_answer[:100], "...")
print("premium:", premium_answer[:100], "...")

| Key | Typical use |
|-----|-------------|
| **`economy` / `premium`** | Cost vs quality routing |
| **`tenant_model`** | Per-customer model selection |
| **`eval_model`** | Fixed model for benchmarks |

Combine with **`init_chat_model`** (**03**) for provider-agnostic alternative maps.

---

## 7. Configurable Prompts — partial() vs configurable_fields

**`.partial()`** freezes variables at chain build time. **`configurable_fields`** freezes them **per invoke**:

In [ ]:
flex_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a {persona}. Reply in {language}."),
    ("human", "{question}"),
])

configurable_prompt = flex_prompt.configurable_fields(
    persona=ConfigurableField(id="persona"),
    language=ConfigurableField(id="language"),
)

persona_chain = configurable_prompt | primary_llm | StrOutputParser()

print(persona_chain.invoke(
    {"question": "Explain pytest fixtures."},
    config={"configurable": {"persona": "senior backend mentor", "language": "English"}},
))
print("---")
print(persona_chain.invoke(
    {"question": "Explain pytest fixtures."},
    config={"configurable": {"persona": "concise docs bot", "language": "English"}},
))

Use **`.partial()`** (**04**) when values never change per deployment; use **`configurable_fields`** when they vary per request or tenant.

---

## 8. Configurable Retriever k (RAG)

Tune retrieval without rebuilding the chain — wrap retriever config or use a lambda:

In [ ]:
DOCS = {
    "c1": "FastAPI routes live under /notes.",
    "c2": "Run alembic upgrade head for migrations.",
    "c3": "JWT uses Authorization: Bearer header.",
}


def search_docs(query: str, k: int = 2) -> str:
    q = query.lower()
    hits = [f"[{cid}] {text}" for cid, text in DOCS.items() if q in text.lower() or q in cid]
    return "\n".join(hits[:k]) or "(no context)"


def configurable_retrieve(query: str, config) -> str:
    k = config.get("configurable", {}).get("retrieval_k", 2)
    return search_docs(query, k=k)


from langchain_core.runnables import RunnableParallel

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using ONLY this context:\n{context}"),
    ("human", "{question}"),
])

configurable_rag = (
    RunnableParallel(
        context=RunnableLambda(configurable_retrieve),
        question=RunnablePassthrough(),
    )
    | rag_prompt
    | primary_llm
    | StrOutputParser()
)

print(configurable_rag.invoke(
    "JWT header",
    config={"configurable": {"retrieval_k": 1}},
))

Pattern from **11. RAG with LCEL** — expose **`retrieval_k`**, **`filter`**, or **`tenant_id`** via `configurable` for per-request tuning.

---

## 9. Parser Fallbacks

Structured parsers (**05**) can raise **`OutputParserException`**. Add a string fallback parser:

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.exceptions import OutputParserException
from langchain_core.language_models.fake_chat_models import FakeListChatModel

json_prompt = ChatPromptTemplate.from_template(
    "Return JSON with keys title and tags for: {text}"
)

json_parser = JsonOutputParser()

# Fake model returns invalid JSON to trigger parser failure
fake_bad_json = FakeListChatModel(responses=["Sure, here is your data: not valid json"])

parsing_chain = json_prompt | fake_bad_json | json_parser

safe_chain = parsing_chain.with_fallbacks([
    json_prompt | fake_bad_json | StrOutputParser(),
])

try:
    parsing_chain.invoke({"text": "a note about migrations"})
except OutputParserException as exc:
    print("primary failed:", type(exc).__name__)

print("fallback:", safe_chain.invoke({"text": "a note about migrations"}))

Production pattern: primary **`with_structured_output`** → fallback to prose + post-hoc validation, or retry with a stricter prompt.

---

## 10. RunnableBranch — Default Route as Fallback

**06. LCEL Composition Patterns** routes by condition. Always provide a **default branch** when the router fails:

In [ ]:
from langchain_core.runnables import RunnableBranch

def route_kind(inputs: dict) -> str:
    q = inputs["question"].lower()
    if any(w in q for w in ("add", "plus", "sum")):
        return "math"
    if any(w in q for w in ("jwt", "auth", "alembic", "pytest")):
        return "docs"
    return "general"


math_chain = RunnableLambda(lambda d: "Use the add tool for arithmetic.")
docs_chain = RunnableLambda(lambda d: "Try search_notes or get_note.")
general_chain = RunnableLambda(lambda d: "Ask a backend documentation question.")

router = RunnableBranch(
    (lambda d: route_kind(d) == "math", math_chain),
    (lambda d: route_kind(d) == "docs", docs_chain),
    general_chain,  # default fallback branch
)

for q in ["What is 2 plus 2?", "JWT header?", "Hello there"]:
    print(q, "→", router.invoke({"question": q}))

Routing is **configurability** (pick subchain); the final **`general_chain`** is a **fallback** when no rule matches.

---

## 11. Agent Model Fallback Middleware

Agents (**13**) support **`ModelFallbackMiddleware`** — try backup models inside the agent loop:

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool


@tool
def lookup(topic: str) -> str:
    """Look up alembic or jwt docs."""
    return "Run alembic upgrade head." if "alembic" in topic.lower() else "Use Bearer header."


fallback_agent = create_agent(
    model=ChatOpenAI(model="gpt-4o", api_key=OPENAI_API_KEY, temperature=0),
    tools=[lookup],
    system_prompt="Use lookup for doc questions.",
    middleware=[
        ModelFallbackMiddleware(
            ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0),
        ),
    ],
)

result = fallback_agent.invoke({
    "messages": [HumanMessage(content="Look up alembic migration command.")],
})
print(result["messages"][-1].content)

Chain-level **`.with_fallbacks([backup_llm])`** and agent **`ModelFallbackMiddleware`** solve the same problem at different layers — use middleware when the agent graph owns model calls.

---

## 12. Combining Config + Fallbacks + Observability

Production chains stack patterns from this notebook and **15**:

In [ ]:
from langchain_core.runnables import RunnableConfig

production_ready = (
    routed_chain
    .with_retry(stop_after_attempt=2, wait_exponential_jitter=True)
    .with_fallbacks([
        RunnableLambda(lambda _: "Service busy — please try again shortly."),
    ])
    .with_config({
        "run_name": "notes_api_qa",
        "tags": ["production", "qa"],
        "metadata": {"service": "notes-api"},
    })
)

answer = production_ready.invoke(
    {"question": "What is JWT?"},
    config=RunnableConfig(configurable={"llm": "economy"}),
)
print(answer)

Order matters: **`with_retry`** on the inner runnable, **`with_fallbacks`** outside retries, **`with_config`** for stable trace labels.

---

## 13. FastAPI Request Pattern (Sketch)

Pass tenant and model tier from the HTTP layer:

In [ ]:
def handle_question(user_tier: str, question: str) -> str:
    llm_key = "premium" if user_tier == "pro" else "economy"
    config = RunnableConfig(
        configurable={"llm": llm_key, "temperature": 0},
        metadata={"user_tier": user_tier},
        tags=["api", user_tier],
    )
    return production_ready.invoke({"question": question}, config=config)


print(handle_question("pro", "What is Alembic?")[:80], "...")
print(handle_question("free", "What is Alembic?")[:80], "...")

---

## 14. Decision Guide

| Need | Use |
|------|-----|
| Same step, transient failure | **`.with_retry()`** |
| Primary model/provider down | **`.with_fallbacks([backup_llm])`** |
| User-facing outage message | **`.with_fallbacks([static_lambda])`** |
| Per-request temperature / max_tokens | **`configurable_fields`** |
| Per-tenant model tier | **`configurable_alternatives`** |
| JSON parse failures | Parser **`.with_fallbacks`** or retry prompt |
| Agent model failures | **`ModelFallbackMiddleware`** |
| Route unknown intent | **`RunnableBranch`** default branch (**06**) |

---

## 15. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Fallback without logging | Silent quality drop | Tag traces; log fallback activation (**15**) |
| Retry on logic errors | Wasted tokens | `retry_if_exception_type` limit to network/rate |
| Infinite retry loops | Cost spike | Cap `stop_after_attempt` |
| Same model as primary and backup | No real fallback | Use distinct model/provider |
| Configurable id mismatch | `KeyError` at invoke | Match `ConfigurableField(id=...)` to config keys |
| Static fallback for all errors | Hides bugs | Fallback only on provider errors |
| Rebuilding chain per tenant | Memory churn | One chain + `configurable_alternatives` |
| Parser fallback returns unvalidated text | Downstream breaks | Validate fallback output shape |

---

## 16. Summary

**Resilience** — **`.with_retry()`** handles transients; **`.with_fallbacks()`** handles hard failures. **Configurability** — **`ConfigurableField`** and **`configurable_alternatives`** swap parameters and implementations via **`RunnableConfig.configurable`** without duplicating chains.

Key takeaways:

- Stack **retry → fallback → static message** for robust APIs.
- **`configurable_alternatives`** routes economy vs premium models per request.
- **Parser fallbacks** recover from malformed structured output (**05**).
- **`RunnableBranch`** needs a default route (**06**).
- **Agents** use **`ModelFallbackMiddleware`** for model-level backup (**13**).
- Combine with **`with_config`** tags/metadata for observability (**15**).

Demonstrations covered lambda and LLM fallbacks, retry policies, configurable temperature and model tiers, configurable RAG retrieval, parser fallbacks, routing defaults, agent model fallback middleware, and a production-ready stacked chain.

Next: **17. Production Patterns for LangChain** — assembling chains, agents, memory, and observability into deployable applications.